In [1]:
import os
import sys
import pandas as pd
from typing import Tuple

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from baseline.turbulence_benchmark.utility.turbulence_log_functions import TurbulenceLogHelper

In [4]:
def compare_multiple_code_generation_logs(res_dir: str, filter: Tuple[str] = (), anti_filter: Tuple[str] = ()):
    
    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (
            os.path.isfile(os.path.join(res_dir, f)) and 
            f.endswith(".csv") and 
            all(sub in f for sub in filter)) and
            all(sub not in f for sub in anti_filter)
            ]

    count = 0

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]

    results_df = pd.DataFrame(columns=log_file_names, index = log_file_names)
    for file_name in log_file_names:
        results_df.loc[file_name, file_name] = float('nan')

    while count < (len(csv_logs)**2-len(csv_logs)//2):
        log1_file_name = csv_logs.pop()
        for log2_file_name in csv_logs:
            log1_file_path = os.path.join(res_dir, log1_file_name)
            log2_file_path = os.path.join(res_dir, log2_file_name)

            log1 = pd.read_csv(log1_file_path)
            log2 = pd.read_csv(log2_file_path) 
            print(log1_file_name, log2_file_name)
            log1_inconsistencies, log2_inconsistencies = TurbulenceLogHelper.compare_turbulence_dataframe_results(log1=log1, log2=log2)

            results_df.loc[log1_file_name.replace('.csv', ''), log2_file_name.replace('.csv', '')] = log1_inconsistencies
            results_df.loc[log2_file_name.replace('.csv', ''), log1_file_name.replace('.csv', '')] = log2_inconsistencies

        count += 1
    
    return results_df

In [5]:
res_dir = proj_dir + "/results/code_generation/gpt-4o"

res = compare_multiple_code_generation_logs(res_dir=res_dir, filter=(), anti_filter=("BigCodeBench", "HumanEval"))
res_df = pd.DataFrame(res)

print(res_df)

Turbulence_zero_shot_random.csv Turbulence_zero_shot_no_mutation.csv
Starting comparison of 52 tasks...

=== COMPARISON SUMMARY ===
Total tasks processed: 52
Both succeeded: 383
Both failed: 0
IdenticalMutationError: 0
Comparable tasks (atleast one succeeded): 433
  - Log1 failed, Log2 succeeded: 29
  - Log1 succeeded, Log2 failed: 21
Total inconsistencies: 50/433
Turbulence_zero_shot_random.csv Turbulence_zero_shot_sequential.csv
Starting comparison of 52 tasks...

=== COMPARISON SUMMARY ===
Total tasks processed: 52
Both succeeded: 404
Both failed: 0
IdenticalMutationError: 0
Comparable tasks (atleast one succeeded): 437
  - Log1 failed, Log2 succeeded: 33
  - Log1 succeeded, Log2 failed: 0
Total inconsistencies: 33/437
Turbulence_zero_shot_sequential.csv Turbulence_zero_shot_no_mutation.csv
Starting comparison of 52 tasks...

=== COMPARISON SUMMARY ===
Total tasks processed: 52
Both succeeded: 410
Both failed: 0
IdenticalMutationError: 0
Comparable tasks (atleast one succeeded): 439